In [11]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

In [12]:
PATH = r"C:\proyectos\red-vial-guadalajara\OneDrive_2026-08-25"
FOLDER_MATRIZ = r"Insumos IMEPLAN\Matriz Origen Destino"
FOLDER_ZONAS = r"red_shapefiles\zones_agebs_visum"

### Leer matriz global de viaje por hora

In [13]:
matriz_global = pd.read_excel(os.path.join(PATH, FOLDER_MATRIZ, "matriz_origen_destino_agebs_por_hora.xlsx"))
matriz_global.head()

,Modo,Hora_inicio,Origen,Destino,Ponderador
0,Transporte Público,0,140980243,1407000211185,9
1,Transporte Público,0,1403900012588,1403900014014,125
2,Transporte Público,0,1403900012624,1403900012639,136
3,Transporte Público,0,1403900012639,1403900012624,136
4,Transporte Público,0,1403900013406,1403900015474,125


In [14]:
print("Dimensiones:", matriz_global.shape)
print("Columnas:", matriz_global.columns.tolist())

print(matriz_global["Modo"].value_counts())

print("\nHORA INICIO")
print("dtype:", matriz_global["Hora_inicio"].dtype)
print("rango:", matriz_global["Hora_inicio"].min(), "a", matriz_global["Hora_inicio"].max())

print("\nLONGITUD CLAVES")
print(matriz_global["Origen"].astype(str).str.len().value_counts())
print(matriz_global["Destino"].astype(str).str.len().value_counts())

Dimensiones: (74643, 5)
Columnas: ['Modo', 'Hora_inicio', 'Origen', 'Destino', 'Ponderador']
Modo
Vehículo Privado      39097
Transporte Público    35546
Name: count, dtype: int64

HORA INICIO
dtype: int64
rango: 0 a 23

LONGITUD CLAVES
Origen
13    69436
9      5207
Name: count, dtype: int64
Destino
13    69502
9      5141
Name: count, dtype: int64


Análisis de los valores de las variables. Observación, de los datos que se deben ajustar creando nuevas columnas de origen y destino con base en un diccionario, varían en sus longitudes. Habrá que revisar si no generan un problema al empatar con el diccionario de ageb.

In [15]:
print("Filas:", len(matriz_global))
print("Claves de origen únicas:", matriz_global["Origen"].nunique())
print("Claves de destino únicas:", matriz_global["Destino"].nunique())

Filas: 74643
Claves de origen únicas: 1660
Claves de destino únicas: 1654


Se validará con los datos de geopandas para ver si sí se pueden empatar todos, tienen que haber más o la misma cantidad de datos

### Map origen y destino al id_ageb_visum

#### Revisión de base de datos geopandas

In [16]:
zones_standarized = gpd.read_file(os.path.join(PATH, FOLDER_ZONAS, "zonas_agebs_standarized.shp"))

print("Dimensiones:", zones_standarized.shape)
print("Columnas:", zones_standarized.columns.tolist())

Dimensiones: (2203, 19)
Columnas: ['clave_ageb', 'clave_enti', 'clave_muni', 'clave_loca', 'ageb', 'nombre_mun', 'tipo_ageb', 'poblacion_', 'area_m2', 'area_km2', 'establecim', 'empleados_', 'densidad_p', 'densidad_e', 'densidad_1', 'distancia_', 'consecutiv', 'id_mun_age', 'geometry']


In [17]:
z = zones_standarized
print("Filas:", len(z))
print("Claves AGEB únicas:", z["clave_ageb"].nunique())
print("AGEB municipales ùnicos:", z["id_mun_age"].nunique())
print("\nNulos:")
print(z[["clave_ageb", "id_mun_age"]].isna().sum())

Filas: 2203
Claves AGEB únicas: 2203
AGEB municipales ùnicos: 2203

Nulos:
clave_ageb    0
id_mun_age    0
dtype: int64


No hay valores nulos y coinciden, por lo que se va a poder empatar bien en el diccionario para la creación de las nuevas columnas.

In [18]:
print("\nLONGITUD CLAVES")
print(z["clave_ageb"].astype(str).str.len().value_counts())
print(z["id_mun_age"].astype(str).str.len().value_counts())


LONGITUD CLAVES
clave_ageb
13    2139
9       64
Name: count, dtype: int64
id_mun_age
4    1760
3     344
2      90
1       9
Name: count, dtype: int64


Vemos que la calve_ageb tiene las mismas dimensiones, es buena señal. id_mun_age estuvo de más, no nos es relevante analizar sus dimensiones.

#### Completado de **matriz_global** con nuevas columnas de origen y destino basadas en el diccionario

In [19]:
print("Claves duplicadas:", z["clave_ageb"].duplicated().sum())

print("\nEjemplos shapefile:", z["clave_ageb"].astype(str).head(3).tolist())
print("Ejemplos matriz   :", matriz_global["Origen"].astype(str).head(3).tolist())

Claves duplicadas: 0

Ejemplos shapefile: ['1403900010026', '1403900010030', '140390001005A']
Ejemplos matriz   : ['140980243', '1403900012588', '1403900012624']


In [20]:
dicc = dict(zip(zones_standarized["clave_ageb"].astype(str),
                zones_standarized["id_mun_age"]))

In [21]:
print(dicc)

{'1403900010026': 1, '1403900010030': 2, '140390001005A': 3, '1403900010064': 4, '1403900010083': 5, '1403900010115': 6, '1403900010134': 7, '1403900010149': 8, '1403900010153': 9, '1403900010191': 10, '1403900010204': 11, '1403900010223': 12, '1403900010238': 13, '1403900010261': 14, '1403900010276': 15, '1403900010280': 16, '1403900010295': 17, '1403900010308': 18, '1403900010312': 19, '1403900010327': 20, '1403900010346': 21, '1403900010350': 22, '1403900010365': 23, '140390001037A': 24, '1403900010384': 25, '1403900010399': 26, '1403900010401': 27, '1403900010416': 28, '1403900010435': 29, '140390001044A': 30, '1403900010454': 31, '1403900010469': 32, '1403900010473': 33, '1403900010492': 34, '1403900010505': 35, '140390001051A': 36, '1403900010524': 37, '1403900010539': 38, '1403900010543': 39, '1403900010562': 40, '1403900010577': 41, '1403900010596': 42, '1403900010609': 43, '1403900010613': 44, '1403900010632': 45, '1403900010647': 46, '1403900010651': 47, '1403900010666': 48, 

El complemento del diciconario con los accesos carreteros es:

```python
accesos_carreteros = {
    10001: 999990004,
    10002: '99999000A',
    10003: 999990003,
    10004: 999990005,
    10005: 999990006,
    10006: 999990001,
    10007: 999990002
}
```

Se le añadirá a nuestro diccionario, pero está invertido el orden, por lo que debemos de voltearlo antes.

In [22]:
accesos_carreteros = {
    10001: 999990004,
    10002: '99999000A',
    10003: 999990003,
    10004: 999990005,
    10005: 999990006,
    10006: 999990001,
    10007: 999990002
}

accesos_carreteros_corregido = {str(clave_ageb): id_mun_age for id_mun_age, clave_ageb in accesos_carreteros.items()}

print(accesos_carreteros_corregido)

{'999990004': 10001, '99999000A': 10002, '999990003': 10003, '999990005': 10004, '999990006': 10005, '999990001': 10006, '999990002': 10007}


In [23]:
dicc.update(accesos_carreteros_corregido)

In [26]:
list(dicc.items())[-8:]

[('141240368', 9054),
 ('999990004', 10001),
 ('99999000A', 10002),
 ('999990003', 10003),
 ('999990005', 10004),
 ('999990006', 10005),
 ('999990001', 10006),
 ('999990002', 10007)]

Vemos que se actualizó el diccionario, ya están todos, próximamente no debería haber valores nulos en las columnas creadas.

In [27]:
matriz_global["Origen_ageb"]  = matriz_global["Origen"].astype(str).map(dicc)
matriz_global["Destino_ageb"] = matriz_global["Destino"].astype(str).map(dicc)

matriz_global.head(100)

,Modo,Hora_inicio,Origen,Destino,Ponderador,Origen_ageb,Destino_ageb
0,Transporte Público,0,140980243,1407000211185,9,6237,3094
1,Transporte Público,0,1403900012588,1403900014014,125,212,288
2,Transporte Público,0,1403900012624,1403900012639,136,216,217
3,Transporte Público,0,1403900012639,1403900012624,136,217,216
4,Transporte Público,0,1403900013406,1403900015474,125,241,422
...,...,...,...,...,...,...,...
95,Transporte Público,3,1407000210736,1403900011359,9,3082,111
96,Transporte Público,3,1409700012098,1409700320284,31,5021,5139
97,Transporte Público,3,1409700250763,1409708311653,54,5094,5297
98,Transporte Público,3,1409700340250,140970180123A,11,5156,5225


In [28]:
print(matriz_global[["Origen_ageb", "Destino_ageb"]].isna().sum())

Origen_ageb     0
Destino_ageb    0
dtype: int64


Verificado, no hay datos NaN, por lo tanto, se empató bien.

### Generar matrices por periodo
- 1 matriz por Modo de transporte x periodo
- Hay 2 modos de transporte: (1) Transporte Público y (2) Vehículo Privado
- Hay 6 periodos: EM, AP, LM, EA, PP, LN
- Generar un total de 12 matrices (tabulares) en base a la hora de inicio del viaje

| Period Code | Period Name | Start Hour | End Hour |
|--------------|----------|-------|-------|
| EM | Early Morning    | 00:00 | 03:59 |
| AP | AM Peak          | 04:00 | 08:59 |
| LM | Late Morning     | 9:00  | 11:59 |
| EA | Early Afternoon  | 12:00 | 17:59 |
| PP | PM Peak          | 18:00 | 20:59 |
| LN | Late Night       | 21:00 | 23:59 |


In [31]:
def asignar_periodo(h):
    if 0 <= h <= 3:
        return "EM"
    elif 4 <= h <= 8:
        return "AP"
    elif 9 <= h <= 11:
        return "LM"
    elif 12 <= h <= 17:
        return "EA"
    elif 18 <= h <= 20:
        return "PP"
    elif 21 <= h <= 23:
        return "LN"

In [32]:
matriz_global["Periodo"] = matriz_global["Hora_inicio"].apply(asignar_periodo)
matriz_global.head()

,Modo,Hora_inicio,Origen,Destino,Ponderador,Origen_ageb,Destino_ageb,Periodo
0,Transporte Público,0,140980243,1407000211185,9,6237,3094,EM
1,Transporte Público,0,1403900012588,1403900014014,125,212,288,EM
2,Transporte Público,0,1403900012624,1403900012639,136,216,217,EM
3,Transporte Público,0,1403900012639,1403900012624,136,217,216,EM
4,Transporte Público,0,1403900013406,1403900015474,125,241,422,EM


In [33]:
base = matriz_global.dropna(subset=["Origen_ageb", "Destino_ageb"]).copy() #se eliminan los Nan con el dropna en esas columnas

In [37]:
print(matriz_global.shape)
print(base.shape)

(74643, 8)
(74643, 8)


Coinciden, el dropna es un vestigio del notebook pasado, pues lo dupliqué, pero no afectó pues no hay NaN ahora en esas dos nuevas columnas.

Vamos a seccionar la tabla con base en *Modo* y *Periodo*, así que las verificamos.

In [38]:
print(len(base))
print(base["Periodo"].value_counts())
print(base["Modo"].value_counts())

74643
Periodo
EA    27351
AP    20428
PP    14411
LM     9006
LN     3062
EM      385
Name: count, dtype: int64
Modo
Vehículo Privado      39097
Transporte Público    35546
Name: count, dtype: int64


In [39]:
print(base["Periodo"].value_counts().sum())
print(base["Modo"].value_counts().sum())

74643
74643


Coinciden en longitud.

### Se crean las tablas y se guardan

In [40]:
## Generar matrices tabulares y guardarlas como xlsx o csv en
OUT_PATH = os.path.join(PATH, "Matriz Origen Destino", "Matrices filtradas por periodo y modo (completas)")

Base filtrada Early Morning y Transporte Público

In [42]:
em_publico = base[(base["Periodo"] == "EM") & (base["Modo"] == "Transporte Público")]
em_publico.to_csv(os.path.join(OUT_PATH, "matriz_TP_EM.csv"), index=False)
print("TP_EM:", len(em_publico))

TP_EM: 112


In [43]:
#úicamente hago una visualización, todas las demás tendrán el mismo formato pues solo es un filtrado
em_publico.head()


,Modo,Hora_inicio,Origen,Destino,Ponderador,Origen_ageb,Destino_ageb,Periodo
0,Transporte Público,0,140980243,1407000211185,9,6237,3094,EM
1,Transporte Público,0,1403900012588,1403900014014,125,212,288,EM
2,Transporte Público,0,1403900012624,1403900012639,136,216,217,EM
3,Transporte Público,0,1403900012639,1403900012624,136,217,216,EM
4,Transporte Público,0,1403900013406,1403900015474,125,241,422,EM


Base filtrada Early Morning y Transporte Privado


In [44]:
em_privado = base[(base["Periodo"] == "EM") & (base["Modo"] == "Vehículo Privado")]
em_privado.to_csv(os.path.join(OUT_PATH, "matriz_VP_EM.csv"), index=False)
print("VP_EM:", len(em_privado))

VP_EM: 273


Base filtrada AM Peak y Transporte Público

In [45]:
ap_publico = base[(base["Periodo"] == "AP") & (base["Modo"] == "Transporte Público")]
ap_publico.to_csv(os.path.join(OUT_PATH, "matriz_TP_AP.csv"), index=False)
print("TP_AP:", len(ap_publico))

TP_AP: 10344


Base filtrada AM Peak y Vehículo Privado

In [46]:
ap_privado = base[(base["Periodo"] == "AP") & (base["Modo"] == "Vehículo Privado")]
ap_privado.to_csv(os.path.join(OUT_PATH, "matriz_VP_AP.csv"), index=False)
print("VP_AP:", len(ap_privado))

VP_AP: 10084


Base filtrada Late Morning y Transporte Público

In [47]:
lm_publico = base[(base["Periodo"] == "LM") & (base["Modo"] == "Transporte Público")]
lm_publico.to_csv(os.path.join(OUT_PATH, "matriz_TP_LM.csv"), index=False)
print("TP_LM:", len(lm_publico))

TP_LM: 4227


Base filtradaLate Morning y Vehículo Privado

In [48]:
lm_privado = base[(base["Periodo"] == "LM") & (base["Modo"] == "Vehículo Privado")]
lm_privado.to_csv(os.path.join(OUT_PATH, "matriz_VP_LM.csv"), index=False)
print("VP_LM:", len(lm_privado))

VP_LM: 4779


Base filtrada Early Afternoon y Transporte Público

In [49]:
ea_publico = base[(base["Periodo"] == "EA") & (base["Modo"] == "Transporte Público")]
ea_publico.to_csv(os.path.join(OUT_PATH, "matriz_TP_EA.csv"), index=False)
print("TP_EA:", len(ea_publico))

TP_EA: 11866


Base filtrada Early Afternoon y Vehículo Privado

In [50]:
ea_privado = base[(base["Periodo"] == "EA") & (base["Modo"] == "Vehículo Privado")]
ea_privado.to_csv(os.path.join(OUT_PATH, "matriz_VP_EA.csv"), index=False)
print("VP_EA:", len(ea_privado))

VP_EA: 15485


Base filtrada PM Peak y Transporte Público

In [51]:
pp_publico = base[(base["Periodo"] == "PP") & (base["Modo"] == "Transporte Público")]
pp_publico.to_csv(os.path.join(OUT_PATH, "matriz_TP_PP.csv"), index=False)
print("TP_PP:", len(pp_publico))

TP_PP: 7873


Base filtrada PM Peak y Vehículo Privado

In [52]:
pp_privado = base[(base["Periodo"] == "PP") & (base["Modo"] == "Vehículo Privado")]
pp_privado.to_csv(os.path.join(OUT_PATH, "matriz_VP_PP.csv"), index=False)
print("VP_PP:", len(pp_privado))

VP_PP: 6538


Base filtrada Late Night y Transporte Público

In [53]:
ln_publico = base[(base["Periodo"] == "LN") & (base["Modo"] == "Transporte Público")]
ln_publico.to_csv(os.path.join(OUT_PATH, "matriz_TP_LN.csv"), index=False)
print("TP_LN:", len(ln_publico))

TP_LN: 1124


Base filtrada Late Night y Vehículo Privado

In [54]:
ln_privado = base[(base["Periodo"] == "LN") & (base["Modo"] == "Vehículo Privado")]
ln_privado.to_csv(os.path.join(OUT_PATH, "matriz_VP_LN.csv"), index=False)
print("VP_LN:", len(ln_privado))

VP_LN: 1938


In [55]:
print(len(os.listdir(OUT_PATH)), "archivos")
print(sorted(os.listdir(OUT_PATH)))

12 archivos
['matriz_TP_AP.csv', 'matriz_TP_EA.csv', 'matriz_TP_EM.csv', 'matriz_TP_LM.csv', 'matriz_TP_LN.csv', 'matriz_TP_PP.csv', 'matriz_VP_AP.csv', 'matriz_VP_EA.csv', 'matriz_VP_EM.csv', 'matriz_VP_LM.csv', 'matriz_VP_LN.csv', 'matriz_VP_PP.csv']


#### Validación

In [56]:
suma = (len(em_publico) + len(em_privado) +
        len(ap_publico) + len(ap_privado) +
        len(lm_publico) + len(lm_privado) +
        len(ea_publico) + len(ea_privado) +
        len(pp_publico) + len(pp_privado) +
        len(ln_publico) + len(ln_privado))

print("Suma de las 12:", suma)
print("Filas en base :", len(base))
print("Filas en matriz_global:", len(matriz_global))
print("Diferencia:", len(matriz_global) - suma)

Suma de las 12: 74643
Filas en base : 74643
Filas en matriz_global: 74643
Diferencia: 0


In [57]:
print("Descartadas por dropna:", 
      (matriz_global["Origen_ageb"].isna() | matriz_global["Destino_ageb"].isna()).sum())

Descartadas por dropna: 0


Coinciden los valores, así que todo bien.